1. Criação do Catálogo

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao;

2. Criação do SCHEMA da Bronze Layer

In [0]:
%sql
USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS bronze;

3. Criação do Volume

In [0]:
%sql
USE SCHEMA bronze;
CREATE VOLUME IF NOT EXISTS landing

4. Validação do Volume

In [0]:
# Validação do Volume

from pyspark.sql.types import StructType

path_landing = "/Volumes/medalhao/bronze/landing/"

# Validação do repo landing
try:
    landing_files = dbutils.fs.ls(path_landing)
    print(f"Conexão com a Landing Zone estabelecida. Total de arquivos identificados: {len(landing_files)}")
except Exception as e:
    print(f"ERRO: Falha ao acessar o Volume 'landing': {e}")

# Declarando
catalogo = "medalhao"
bronze_db = "bronze"
volume_landing = "landing"

Conexão com a Landing Zone estabelecida. Total de arquivos identificados: 5


5. Widgets de Texto

In [0]:
# Criando Widgets de texto
dbutils.widgets.text("data_inicio", "09-01-2016")
dbutils.widgets.text("data_fim", "12-31-2018")

# Lê os valores informados nos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print("Data início:", data_inicio)
print("Data fim:", data_fim)

Data início: 09-01-2016
Data fim: 12-31-2026


6. Conectando à API do Banco Central

In [0]:
import requests
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicio}'",
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

# Tenta buscar os dados na API com tratamento de erro e resiliência de rede
try:
    response = requests.get(url, params=params, timeout=10)
    if response.status_code == 200:
        dados_json = response.json()
        dados_cotacao = dados_json.get("value", [])
    else:
        dados_cotacao = []
except Exception as e:
    print(f"Aviso: Não foi possível acessar a API do Banco Central no momento ({e}). Criando tabela com esquema oficial.")
    dados_cotacao = []

# Schema oficial para garantir os tipos de dados
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

# Converte a lista em DataFrame Spark
if dados_cotacao:
    df_cotacao_raw = spark.createDataFrame(dados_cotacao)
else:
    df_cotacao_raw = spark.createDataFrame([], schema=schema_cotacao)

# Adiciona timestamp de auditoria
df_cotacao = df_cotacao_raw.withColumn("ingestion_datetime", current_timestamp())

# Salva como tabela Delta na Bronze (tb_cotacao_dolar)
(
    df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{bronze_db}.tb_cotacao_dolar")
)

print("Tabela 'medalhao.bronze.tb_cotacao_dolar' salva com SUCESSO!")

Tabela 'medalhao.bronze.tb_cotacao_dolar' salva com SUCESSO!


7. Ingerindo Dados

In [0]:
mapeamento_bronze = {
    "movies_info_TMDB_IMDB.csv": f"{catalogo}.{bronze_db}.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": f"{catalogo}.{bronze_db}.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": f"{catalogo}.{bronze_db}.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": f"{catalogo}.{bronze_db}.tb_credits_and_tags",
    "movies_reviews.csv": f"{catalogo}.{bronze_db}.tb_movies_reviews"
}

for arquivo_csv, tabela_destino in mapeamento_bronze.items():
    print(f"Ingerindo '{arquivo_csv}' > Tabela: '{tabela_destino}'...")
    
    # Read mantendo fidelidade aos dados brutos
    df_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(f"{path_landing}{arquivo_csv}")
    )

    # Injeção da coluna de auditoria ingestion_datetime
    df_bronze = df_raw.withColumn("ingestion_datetime", current_timestamp())

    # Persistência na camada Bronze em formato Delta
    df_bronze.write.format("delta").mode("append").saveAsTable(tabela_destino)

print("Ingestão de 5 arquivos CSV concluída com SUCESSO")

Ingerindo 'movies_info_TMDB_IMDB.csv' > Tabela: 'medalhao.bronze.tb_movies_info'...
Ingerindo 'movies_financials_IMDB_TMDB.csv' > Tabela: 'medalhao.bronze.tb_movies_financials'...
Ingerindo 'movies_metrics_IMDB_TMDB.csv' > Tabela: 'medalhao.bronze.tb_movies_metrics'...
Ingerindo 'credits_and_tags_IMDB_TMDB.csv' > Tabela: 'medalhao.bronze.tb_credits_and_tags'...
Ingerindo 'movies_reviews.csv' > Tabela: 'medalhao.bronze.tb_movies_reviews'...
Ingestão de 5 arquivos CSV concluída com SUCESSO
